# Jacobian sanity check

`mj_jacSite` fills a `3 x nv` row block covering every degree of freedom in the model, not just the arm. The Panda scene has `nv = 9`: seven arm joints plus two gripper fingers. The solver uses `dof_ids` to pick the seven arm columns out of that block.

If `dof_ids` selects the wrong columns, the solver still runs and still produces motion that looks plausible on screen. Nothing downstream fails loudly. The only reliable way to catch it is to compare the analytic Jacobian against a numerical one.

The check: perturb one `qpos` entry, call `mj_forward`, measure how far the site moved, and divide by the step. That ratio is the corresponding Jacobian column.

The same comparison runs in `tests/test_jacobian.py` with a `1e-4` tolerance.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mujoco
import numpy as np

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from diffik import model as robot_model

handles = robot_model.load(REPO_ROOT / "scene" / "panda_ik.xml")
m, d = handles.model, handles.data

print(f"nq={m.nq}  nv={m.nv}  nu={m.nu}")
print(f"arm dof_ids  = {handles.dof_ids}")
print(f"arm qpos_ids = {handles.qpos_ids}")
print(f"site_xpos    = {d.site_xpos[handles.site_id]}")

## Analytic Jacobian

`mj_jacSite` writes the linear block into `jacp` and the angular block into `jacr`. Stacking them gives the `6 x nv` matrix; slicing with `dof_ids` reduces it to the `6 x 7` matrix the solver actually works with.

In [ ]:
def set_arm(q_arm):
    """Write an arm configuration and refresh the derived quantities."""
    d.qpos[handles.qpos_ids] = q_arm
    mujoco.mj_forward(m, d)


def analytic_jacobian():
    jacp = np.zeros((3, m.nv))
    jacr = np.zeros((3, m.nv))
    mujoco.mj_jacSite(m, d, jacp, jacr, handles.site_id)
    return np.vstack([jacp[:, handles.dof_ids], jacr[:, handles.dof_ids]])


robot_model.reset_to_home(handles)
J = analytic_jacobian()
print(f"J shape {J.shape}, condition number {np.linalg.cond(J):.1f}")
np.set_printoptions(precision=4, suppress=True)
J

## Numerical Jacobian

The linear block is a plain difference of site positions. The angular block is not: orientations do not subtract. The relative rotation between the two quaternions is converted to a rotation vector with `mju_quat2Vel`, which is the same quantity `jacr` reports.

Both a forward and a central difference are computed. The forward version is the obvious one; the central version cancels the first-order truncation term and is what the test uses.

In [ ]:
EPS = 1e-6


def site_pose():
    pos = d.site_xpos[handles.site_id].copy()
    quat = np.zeros(4)
    mujoco.mju_mat2Quat(quat, d.site_xmat[handles.site_id])
    return pos, quat


def rotation_between(quat_from, quat_to):
    """Rotation vector taking quat_from to quat_to."""
    inverse = np.zeros(4)
    relative = np.zeros(4)
    rotation = np.zeros(3)
    mujoco.mju_negQuat(inverse, quat_from)
    mujoco.mju_mulQuat(relative, quat_to, inverse)
    mujoco.mju_quat2Vel(rotation, relative, 1.0)
    return rotation


def numerical_jacobian(q_arm, central=True):
    jac = np.zeros((6, 7))
    for column in range(7):
        q_plus = q_arm.copy()
        q_plus[column] += EPS
        set_arm(q_plus)
        pos_plus, quat_plus = site_pose()

        if central:
            q_minus = q_arm.copy()
            q_minus[column] -= EPS
            set_arm(q_minus)
            step = 2.0 * EPS
        else:
            set_arm(q_arm)
            step = EPS
        pos_ref, quat_ref = site_pose()

        jac[:3, column] = (pos_plus - pos_ref) / step
        jac[3:, column] = rotation_between(quat_ref, quat_plus) / step

    set_arm(q_arm)
    return jac


q_home = handles.q_home.copy()
set_arm(q_home)

J_analytic = analytic_jacobian()
J_central = numerical_jacobian(q_home, central=True)
J_forward = numerical_jacobian(q_home, central=False)

print(f"central max abs error {np.abs(J_analytic - J_central).max():.3e}")
print(f"forward max abs error {np.abs(J_analytic - J_forward).max():.3e}")

## Per-column error across several postures

One posture is not enough: a wrong column index can coincidentally match at a configuration where two joints happen to produce similar motion. The last posture sits close to full extension, where the Jacobian is badly conditioned.

In [ ]:
CONFIGURATIONS = {
    "home": np.array([0.0, 0.0, 0.0, -1.57079, 0.0, 1.57079, -0.7853]),
    "mid A": np.array([0.3, -0.4, 0.2, -1.9, 0.5, 1.2, -0.3]),
    "mid B": np.array([-0.8, 0.6, -0.5, -0.9, -0.7, 2.1, 0.9]),
    "near extension": np.array([1.2, 0.9, 1.0, -0.3, 1.4, 3.0, 1.8]),
}

errors = {}
conditions = {}
for label, q in CONFIGURATIONS.items():
    set_arm(q)
    J_a = analytic_jacobian()
    J_n = numerical_jacobian(q, central=True)
    errors[label] = np.linalg.norm(J_a - J_n, axis=0)
    conditions[label] = np.linalg.cond(J_a)

for label, err in errors.items():
    print(f"{label:>15}  max column error {err.max():.3e}  cond(J) {conditions[label]:8.1f}")

In [ ]:
TOLERANCE = 1e-4
joints = np.arange(1, 8)
width = 0.2

fig, ax = plt.subplots(figsize=(9, 4.5))
for offset, (label, err) in enumerate(errors.items()):
    ax.bar(joints + (offset - 1.5) * width, err, width, label=label)

ax.axhline(TOLERANCE, color="black", linestyle="--", linewidth=1,
           label=f"test tolerance {TOLERANCE:g}")
ax.set_yscale("log")
ax.set_xlabel("arm joint")
ax.set_ylabel("column error norm")
ax.set_title("Analytic vs finite-difference site Jacobian, per column")
ax.set_xticks(joints)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Finger columns

The two gripper dofs slide the fingers along the hand. They cannot move the hand itself, so their Jacobian columns are exactly zero. This is why they are dropped from `dof_ids` rather than merely left in place: keeping them would make `J J^T` singular in that direction and force the damping to do work it should not have to do.

In [ ]:
robot_model.reset_to_home(handles)
jacp = np.zeros((3, m.nv))
jacr = np.zeros((3, m.nv))
mujoco.mj_jacSite(m, d, jacp, jacr, handles.site_id)

finger_dofs = [
    m.jnt_dofadr[mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_JOINT, name)]
    for name in ("finger_joint1", "finger_joint2")
]
print(f"finger dof indices {finger_dofs}")
print(f"max |jacp| on finger columns {np.abs(jacp[:, finger_dofs]).max():.3e}")
print(f"max |jacr| on finger columns {np.abs(jacr[:, finger_dofs]).max():.3e}")

## Conclusion

All seven columns agree with the finite-difference estimate far below the `1e-4` tolerance, at every posture tested. `dof_ids` selects the right columns, and the Jacobian the solver receives describes the motion the arm actually makes.

If a column ever disagrees, the fault is in index resolution in `diffik/model.py`, not in MuJoCo. Fix that before trusting anything downstream.